# Evaluation notebook
This notebook evaluates a dense embedding baseline using the same binary relevance setting as the existing TF-IDF, Qwen, and MiniLM notebooks.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import json
import numpy as np
from sentence_transformers import SentenceTransformer

from utils import (
    avg_prf_at_k,
    avg_mrr_at_k,
    avg_hits_at_k,
    avg_jaccard_at_k,
    avg_ndcg_at_k,
)

MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
BATCH_SIZE = 32
TOPKS = [3, 5, 10, 20]


/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the triplet dataset
file_path = "hard_triplets_tfidf_wikidata.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Successfully loaded {len(data)} query groups from the JSON file.")


Successfully loaded 186 query groups from the JSON file.


In [3]:
# Build evaluation items and unique query vocabulary
unique_queries = set()
eval_items = []

for item in data:
    anchor = item["anchor_query"]
    positives = item["positive_queries"]
    negatives = item["negative_queries"]

    pool = positives + negatives

    unique_queries.add(anchor)
    unique_queries.update(positives)
    unique_queries.update(negatives)
    eval_items.append((anchor, pool, positives))

queries_list = list(unique_queries)
print(f"Extracted {len(eval_items)} evaluation items.")
print(f"Found {len(queries_list)} unique queries for the embedding vocabulary.")


Extracted 186 evaluation items.
Found 2227 unique queries for the embedding vocabulary.


In [4]:
# Build binary relevance maps from annotated positives
binary_relevance_per_item = []
for _, pool, positives in eval_items:
    positive_set = set(positives)
    relevance_map = {q: (1.0 if q in positive_set else 0.0) for q in pool}
    binary_relevance_per_item.append(relevance_map)

print(f"Built binary relevance maps for {len(binary_relevance_per_item)} items.")


Built binary relevance maps for 186 items.


In [5]:
# Dense model helper functions
def build_emb_map(model, queries, batch_size=32, transform=None, normalize_embeddings=True):
    emb_map = {}

    for i in range(0, len(queries), batch_size):
        batch = queries[i:i + batch_size]
        texts = [transform(q) for q in batch] if transform else batch
        embs = model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=normalize_embeddings,
        )

        for q, e in zip(batch, embs):
            emb_map[q] = e

    return emb_map


def build_sorted_pools_from_emb_map(emb_map, items):
    sorted_pools = []

    for anchor_q, pool, _ in items:
        sims = np.vstack([emb_map[q] for q in pool]) @ emb_map[anchor_q]
        order = np.argsort(-sims)
        sorted_pools.append([pool[i] for i in order])

    return sorted_pools


In [6]:
print(f"Loading {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

emb_map = build_emb_map(
    model,
    queries_list,
    batch_size=BATCH_SIZE,
    transform=None,
)

sorted_pools = build_sorted_pools_from_emb_map(emb_map, eval_items)


Loading sentence-transformers/all-mpnet-base-v2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1061.73it/s, Materializing param=pooler.dense.weight]                       
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
print("\n" + "=" * 60)
print(f"FINAL EVALUATION RESULTS: {MODEL_NAME}")
print("=" * 60)

print("\n[Binary relevance: Annotated Positives]")
print(f"{'':5} {'P':>7} {'R':>7} {'F1':>7} {'MRR':>7} {'Hits':>7} {'avgJacc':>9} {'nDCG':>7}")
print("-" * 62)

for k in TOPKS:
    p, r, f1 = avg_prf_at_k(sorted_pools, eval_items, k)
    mrr = avg_mrr_at_k(sorted_pools, eval_items, k)
    hits = avg_hits_at_k(sorted_pools, eval_items, k)
    avg_jacc = avg_jaccard_at_k(sorted_pools, k, binary_relevance_per_item)
    ndcg = avg_ndcg_at_k(sorted_pools, k, binary_relevance_per_item)

    print(f"@{k:<4} {p:>7.4f} {r:>7.4f} {f1:>7.4f} {mrr:>7.4f} {hits:>7.4f} {avg_jacc:>9.4f} {ndcg:>7.4f}")



FINAL EVALUATION RESULTS: sentence-transformers/all-mpnet-base-v2

[Binary relevance: Annotated Positives]
            P       R      F1     MRR    Hits   avgJacc    nDCG
--------------------------------------------------------------
@3     0.2993  0.0795  0.1181  0.3477  0.4677    0.2993  0.2901
@5     0.3129  0.1397  0.1777  0.3748  0.5860    0.3140  0.3064
@10    0.3215  0.3141  0.2815  0.3971  0.7634    0.3247  0.3502
@20    0.3306  0.6480  0.3881  0.4117  0.9731    0.3359  0.4750
